<a href="https://colab.research.google.com/github/Linqiaoqiao2/xAI_Project_DG_Test-Time-Augmentation/blob/main/dg_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install torchvision
!pip install timm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 124.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 58.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights
from torchvision import models
from torch.utils.data import DataLoader, Dataset
import os
from torchvision import datasets, transforms



In [ ]:
#!/usr/bin/env python3
"""
dg_pacs_pipeline.py

Pipeline for domain generalization on the PACS dataset.

Usage in Notebook/Colab:
    # 1. Mount your Drive and ensure PACS.zip is in one of the default locations
    # 2. Run:
    #    python dg_pacs_pipeline.py [--base_dir /content/PACS/kfold] [--epochs N]

CLI Usage:
    python dg_pacs_pipeline.py [--base_dir /path/to/PACS/kfold] [--epochs N]

Ensure:
    - PyTorch and torchvision are installed: pip install torch torchvision
    - PACS.zip exists in default Colab paths or specify with --zip_path
"""
import os
import sys
import argparse
import zipfile

# Ensure PyTorch is installed
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, ConcatDataset
except ModuleNotFoundError:
    sys.exit("Error: PyTorch is not installed. Please install it with `pip install torch torchvision`.")

from torchvision import datasets, transforms, models


def ensure_dataset_unzipped(base_dir: str, zip_path: str):
    """
    If base_dir doesn't exist, search for PACS.zip in common Colab locations and unzip it.
    """
    if os.path.isdir(base_dir):
        return
    # Candidate ZIP locations
    candidates = [
        zip_path,
        '/content/drive/MyDrive/Colab Notebooks/PACS.zip',
        '/content/drive/MyDrive/PACS.zip',
        '/content/PACS.zip'
    ]
    found = None
    for zp in candidates:
        if zp and os.path.isfile(zp):
            found = zp
            break
    if not found:
        sys.exit(f"Error: base_dir '{base_dir}' not found and no PACS.zip in {candidates}.")
    print(f"Unzipping dataset from {found} to {os.path.dirname(base_dir)}...")
    with zipfile.ZipFile(found, 'r') as z:
        z.extractall(os.path.dirname(base_dir))
    if not os.path.isdir(base_dir):
        sys.exit(f"Error: after unzipping, base_dir '{base_dir}' still not found.")


def get_model(name: str, num_classes: int, device: torch.device) -> torch.nn.Module:
    """
    Return a ResNet model ('resnet18' or 'resnet50') with final FC layer adapted.
    """
    if name == 'resnet18':
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    elif name == 'resnet50':
        base = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    else:
        raise ValueError(f"Unknown model: {name}")
    in_features = base.fc.in_features
    base.fc = nn.Linear(in_features, num_classes)
    return base.to(device)


def train_one_epoch(model: torch.nn.Module,
                    loader: DataLoader,
                    optimizer: torch.optim.Optimizer,
                    criterion: torch.nn.Module,
                    device: torch.device) -> float:
    model.train()
    total_loss = 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
    return total_loss / len(loader.dataset)


def evaluate(model: torch.nn.Module,
             loader: DataLoader,
             device: torch.device) -> float:
    model.eval()
    correct = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
    return correct / len(loader.dataset)


def domain_generalization_pipeline(
    base_dir: str,
    model_name: str,
    num_classes: int = 7,
    batch_size: int = 32,
    num_epochs: int = 5,
    lr_list = [1e-3, 5e-4],
    wd_list = [1e-4, 5e-5],
    zip_path: str = '/content/PACS.zip'
) -> None:
    # Ensure data available
    ensure_dataset_unzipped(base_dir, zip_path)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    domains = sorted([d for d in os.listdir(base_dir)
                      if os.path.isdir(os.path.join(base_dir, d))])
    if not domains:
        sys.exit(f"Error: no domain folders found in {base_dir}.")
    print(f"Found domains: {domains}")

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    criterion = nn.CrossEntropyLoss()
    results = {}

    for test_domain in domains:
        print(f"\n=== Testing on domain: {test_domain} ===")
        sources = [d for d in domains if d != test_domain]

        # Hyperparameter selection
        best_cfg, best_worst = None, -float('inf')
        for lr in lr_list:
            for wd in wd_list:
                worst_vals = []
                for val in sources:
                    train_src = [d for d in sources if d != val]
                    ds_tr = ConcatDataset([
                        datasets.ImageFolder(os.path.join(base_dir, d), transform)
                        for d in train_src
                    ])
                    ds_val = datasets.ImageFolder(os.path.join(base_dir, val), transform)
                    ltr = DataLoader(ds_tr, batch_size=batch_size, shuffle=True)
                    lval = DataLoader(ds_val, batch_size=batch_size, shuffle=False)
                    m = get_model(model_name, num_classes, device)
                    o = optim.Adam(m.parameters(), lr=lr, weight_decay=wd)
                    for _ in range(3):
                        train_one_epoch(m, ltr, o, criterion, device)
                    worst_vals.append(evaluate(m, lval, device))
                worst = min(worst_vals)
                print(f"  cfg lr={lr}, wd={wd} -> worst={worst:.4f}")
                if worst > best_worst:
                    best_worst, best_cfg = worst, (lr, wd)
        print(f"Selected lr={best_cfg[0]}, wd={best_cfg[1]}, worst={best_worst:.4f}")

        # Final train & test
        ds_src = ConcatDataset([
            datasets.ImageFolder(os.path.join(base_dir, d), transform)
            for d in sources
        ])
        ds_test = datasets.ImageFolder(os.path.join(base_dir, test_domain), transform)
        l_src = DataLoader(ds_src, batch_size=batch_size, shuffle=True)
        l_test = DataLoader(ds_test, batch_size=batch_size, shuffle=False)
        mf = get_model(model_name, num_classes, device)
        of = optim.Adam(mf.parameters(), lr=best_cfg[0], weight_decay=best_cfg[1])
        for e in range(1, num_epochs+1):
            loss = train_one_epoch(mf, l_src, of, criterion, device)
            print(f"  Epoch {e}/{num_epochs} - loss={loss:.4f}")
        acc = evaluate(mf, l_test, device)
        print(f"--> Test acc on {test_domain}: {acc*100:.2f}%")
        results[test_domain] = acc

    print("\n=== Summary ===")
    accs = results.values()
    for d, a in results.items():
        print(f"  {d:12s}: {a*100:.2f}%")
    print(f"Avg: {sum(accs)/len(accs)*100:.2f}% | Worst: {min(accs)*100:.2f}%")


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser()
    p.add_argument('--base_dir', type=str,
                   default=os.getenv('BASE_DIR', '/content/PACS/kfold'),
                   help="Path to PACS kfold folder.")
    p.add_argument('--epochs', type=int, default=5,
                   help="Number of final training epochs.")
    p.add_argument('--zip_path', type=str,
                   default=os.getenv('BASE_DIR_ZIP', '/content/PACS.zip'),
                   help="Optional path to PACS.zip")
    args, _ = p.parse_known_args()
    return args

if __name__ == '__main__':
    args = parse_args()
    for name in ['resnet18', 'resnet50']:
        print(f"\n***** Baseline: {name} *****")
        domain_generalization_pipeline(
            base_dir=args.base_dir,
            model_name=name,
            num_epochs=args.epochs,
            zip_path=args.zip_path
        )



***** Baseline: resnet18 *****
Found domains: ['art_painting', 'cartoon', 'photo', 'sketch']

=== Testing on domain: art_painting ===
  cfg lr=0.001, wd=0.0001 -> worst=0.4246
  cfg lr=0.001, wd=5e-05 -> worst=0.4057
  cfg lr=0.0005, wd=0.0001 -> worst=0.3699
  cfg lr=0.0005, wd=5e-05 -> worst=0.5469
Selected lr=0.0005, wd=5e-05, worst=0.5469
  Epoch 1/5 - loss=0.4662
  Epoch 2/5 - loss=0.2100
  Epoch 3/5 - loss=0.1526
  Epoch 4/5 - loss=0.1081
  Epoch 5/5 - loss=0.0971
--> Test acc on art_painting: 63.67%

=== Testing on domain: cartoon ===
  cfg lr=0.001, wd=0.0001 -> worst=0.3936
  cfg lr=0.001, wd=5e-05 -> worst=0.3762
  cfg lr=0.0005, wd=0.0001 -> worst=0.3521
  cfg lr=0.0005, wd=5e-05 -> worst=0.4897
Selected lr=0.0005, wd=5e-05, worst=0.4897
  Epoch 1/5 - loss=0.5087
  Epoch 2/5 - loss=0.2640
  Epoch 3/5 - loss=0.1790
  Epoch 4/5 - loss=0.0945
  Epoch 5/5 - loss=0.1134
--> Test acc on cartoon: 63.82%

=== Testing on domain: photo ===
  cfg lr=0.001, wd=0.0001 -> worst=0.4419
  

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 144MB/s]


  cfg lr=0.001, wd=0.0001 -> worst=0.2838
  cfg lr=0.001, wd=5e-05 -> worst=0.4778
  cfg lr=0.0005, wd=0.0001 -> worst=0.6114
  cfg lr=0.0005, wd=5e-05 -> worst=0.5472
Selected lr=0.0005, wd=0.0001, worst=0.6114
  Epoch 1/5 - loss=0.4531
  Epoch 2/5 - loss=0.1829
  Epoch 3/5 - loss=0.1430
  Epoch 4/5 - loss=0.0888
  Epoch 5/5 - loss=0.0812
--> Test acc on art_painting: 65.97%

=== Testing on domain: cartoon ===
  cfg lr=0.001, wd=0.0001 -> worst=0.3765
  cfg lr=0.001, wd=5e-05 -> worst=0.3026
  cfg lr=0.0005, wd=0.0001 -> worst=0.5088
